In [1]:
import os
os.chdir("../../")#与easymlops同级目录

### 拆分数据

In [2]:
import pandas as pd
data=pd.read_csv("./data/demo.csv")
data.head(5)

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


In [3]:
x_train=data[:500]
x_test=data[500:]
y_train=x_train["Survived"]
y_test=x_test["Survived"]
del x_train["Survived"]
del x_test["Survived"]

### 简单跑一个模型

In [4]:
from easymlops import TablePipeLine
from easymlops.table.preprocessing import *
from easymlops.table.encoding import *
from easymlops.table.classification import *

In [5]:
table=TablePipeLine()
table.pipe(FixInput())\
  .pipe(FillNa())\
  .pipe(OneHotEncoding(cols=["Pclass", "Sex"], drop_col=False)) \
  .pipe(WOEEncoding(cols=["Ticket", "Embarked", "Cabin", "Sex", "Pclass"], y=y_train)) \
  .pipe(LabelEncoding(cols=["Name"]))\
  .pipe(LGBMClassification(y=y_train,native_init_params={"max_depth":2},native_fit_params={"num_boost_round":128},prefix="lgbm"))

x_test_new=table.fit(x_train).transform(x_test)
x_test_new.head(5)

,lgbm_0,lgbm_1
500,0.923326,0.076674
501,0.373652,0.626348
502,0.378380,0.621620
503,0.670166,0.329834
504,0.068470,0.931530


### 挂载

In [6]:
from easymlops.table.storage import HbaseStorage

In [7]:
#记录label encoding的输出
table[-2].set_branch_pipe(HbaseStorage(host=["192.168.244.131"],port=9090,table_name="label_encoding",cf_name="cf1",cols=['PassengerId','Pclass','Name','Sex','Age','SibSp','Parch',
                                                                             'Ticket','Fare','Cabin','Embarked','Pclass_3','Pclass_1','Pclass_2','Sex_male','Sex_female']))

connect to 192.168.244.131:9090/label_encoding success!


### 模拟线上数据流

In [ ]:
for record in tqdm(x_test.to_dict("record")):
    for index in range(1,1000):
        table.transform_single(record,storage_base_dict={"key":f"{record.get('PassengerId')}_{index}"})

 96%|████████████████████████████████████████████████████████████████████████████▌   | 374/391 [18:39<00:58,  3.46s/it]

### 存储结果分析  
这部分需要特别注意的是：弄清楚你分析数据所需的计算资源是由存储引擎提供还是本地机器，如果是本地机器，尽量避免复杂的查询

###  查询某key对应的记录

In [9]:
table[-2].get_branch_pipe(0).select_key(key=501)

,storage_key,storage_transform_time,PassengerId,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked,Pclass_3,Pclass_1,Pclass_2,Sex_male,Sex_female
0,501,2023-03-10 16:12:49.292222,501,0.4824385226871724,0,1.1113788020761075,17.0,0,0,0.0,8.6640625,0.299607189091587,0.22484882921160226,1,0,0,1,0


### 异步保存
存储模块强制使用异步的方式存储数据，通常只消耗少量transform_single的时间(获取当前时间以及字典中的数据，并放到队列中异步存储)

In [10]:
from easymlops.table.callback import check_transform_function_pipeline
table.callback(check_transform_function_pipeline,x_test[:10])

(<class 'easymlops.table.preprocessing.core.FixInput'>) module check [transform] complete,speed:[0.4ms]/it,cpu:[0%],memory:[0K]
(<class 'easymlops.table.preprocessing.onevar_operation.FillNa'>) module check [transform] complete,speed:[0.0ms]/it,cpu:[0%],memory:[0K]
(<class 'easymlops.table.encoding.OneHotEncoding'>) module check [transform] complete,speed:[0.0ms]/it,cpu:[0%],memory:[0K]
(<class 'easymlops.table.encoding.WOEEncoding'>) module check [transform] complete,speed:[0.0ms]/it,cpu:[0%],memory:[0K]
(<class 'easymlops.table.encoding.LabelEncoding'>) module check [transform] complete,speed:[0.1ms]/it,cpu:[0%],memory:[0K]
(<class 'easymlops.table.classification.LGBMClassification'>) module check [transform] complete,speed:[3.9ms]/it,cpu:[33%],memory:[32K]


In [11]:
table[-2].get_branch_pipe(0).select_key(key="default")

,storage_key,storage_transform_time,PassengerId,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked,Pclass_3,Pclass_1,Pclass_2,Sex_male,Sex_female
0,default,2023-03-10 16:12:55.219446,510,0.4824385226871724,0,1.1113788020761075,26.0,0,0,-0.4641575586823115,56.5,0.299607189091587,0.22484882921160226,1,0,0,1,0


### 持久化测试

In [12]:
table.save("ml.pkl")

In [19]:
table=TablePipeLine()
table.pipe(FixInput())\
  .pipe(FillNa())\
  .pipe(OneHotEncoding(cols=["Pclass", "Sex"], drop_col=False)) \
  .pipe(WOEEncoding(cols=["Ticket", "Embarked", "Cabin", "Sex", "Pclass"])) \
  .pipe(LabelEncoding(cols=["Name"]))\
  .pipe(LGBMClassification(native_init_params={"max_depth":2},native_fit_params={"num_boost_round":128},prefix="lgbm"))


table[-2].set_branch_pipe(HbaseStorage())
table.load("ml.pkl")

connect to localhost:9090/table_name exception:TTransportException(type=1, message="Could not connect to ('localhost', 9090)")
connect to localhost:9090/table_name exception:TTransportException(type=1, message="Could not connect to ('localhost', 9090)")
connect to 192.168.244.131:9090/label_encoding success!


In [14]:
for record in tqdm(x_train[:10].to_dict("record")):
    table.transform_single(record,storage_base_dict={"key":record.get("PassengerId")})

100%|█████████████████████████████████████████████████████████████████████████████████| 10/10 [00:00<00:00, 163.87it/s]


In [20]:
table[-2].get_branch_pipe(0).select_key(key=3)

,storage_key,storage_transform_time,PassengerId,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked,Pclass_3,Pclass_1,Pclass_2,Sex_male,Sex_female
0,3,2023-03-10 16:13:01.829374,3,0.4824385226871724,3,-1.5699900953239083,26.0,0,0,0.0,7.92578125,0.299607189091587,0.22484882921160226,1,0,0,0,1
